# Eval Data Prep

Build evaluation story pairs for embedding-model cosine similarity tests from:
- `tell_me_again_v1`
- `movie_remakes` (or fallback path `MovieRemakeDataset_NAACL2018`)

Target (configurable):
- 100 pairs from Tell Me Again
- 100 pairs from Movie Remakes
- In each dataset: 50% positive (true retell/remake), 50% random negative


In [1]:
import json
import random
from pathlib import Path
import pandas as pd

In [2]:
def find_project_root() -> Path:
    candidates = [Path.cwd().resolve(), Path.cwd().resolve().parent]
    for c in candidates:
        if (c / '.git').exists() and (c / 'data').exists() and (c / 'src').exists():
            return c
    raise RuntimeError('Could not locate project root from current working directory.')

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'data'
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_DIR:', DATA_DIR)


PROJECT_ROOT: /Users/shayan/Projects/NarrativeSimilarity
DATA_DIR: /Users/shayan/Projects/NarrativeSimilarity/data


In [3]:
CONFIG = {
    'seed': 42,
    'tell_me_again_total_pairs': 100,
    'movie_remakes_total_pairs': 100,
    'positive_ratio': 0.5,

    # Paths
    'tell_me_again_dir': DATA_DIR / 'tell_me_again_v1',
    'movie_remakes_dir': DATA_DIR / 'MovieRemakeDataset_NAACL2018',

    # Output
    'output_json': DATA_DIR / 'eval_data' / 'eval_story_pairs_200.json',
    'output_csv': DATA_DIR / 'eval_data' / 'eval_story_pairs_200.csv',
}

random.seed(CONFIG['seed'])

if not CONFIG['movie_remakes_dir'].exists() and CONFIG['movie_remakes_fallback_dir'].exists():
    CONFIG['movie_remakes_dir'] = CONFIG['movie_remakes_fallback_dir']

print('tell_me_again_dir:', CONFIG['tell_me_again_dir'])
print('movie_remakes_dir:', CONFIG['movie_remakes_dir'])
print('output_json:', CONFIG['output_json'])
print('output_csv:', CONFIG['output_csv'])


tell_me_again_dir: /Users/shayan/Projects/NarrativeSimilarity/data/tell_me_again_v1
movie_remakes_dir: /Users/shayan/Projects/NarrativeSimilarity/data/MovieRemakeDataset_NAACL2018
output_json: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.json
output_csv: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv


## Explore Tell Me Again (dev split when available)


In [4]:
tma_dir = CONFIG['tell_me_again_dir']
dev_csv = tma_dir / 'dev_stories.csv'
summaries_root = tma_dir / 'summaries'

if not tma_dir.exists():
    raise FileNotFoundError(f'tell_me_again_v1 folder not found: {tma_dir}')

if not dev_csv.exists():
    raise FileNotFoundError(f'dev_stories.csv not found: {dev_csv}')

dev_ids = pd.read_csv(dev_csv, header=None, names=['wikidata_id'])
dev_ids['wikidata_id'] = dev_ids['wikidata_id'].astype(str).str.strip()

def load_tma_summary_json(wikidata_id: str):
    p = summaries_root / wikidata_id[:2] / f'{wikidata_id}.json'
    if not p.exists():
        return None
    try:
        with open(p, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception:
        return None

def collect_tma_texts(obj: dict):
    texts = []

    # summaries is usually a dict: lang -> summary_text
    s = obj.get('summaries')
    if isinstance(s, dict):
        for lang, val in s.items():
            if isinstance(val, str) and val.strip():
                texts.append((f'summaries:{lang}', val.strip()))
    elif isinstance(s, list):
        for i, val in enumerate(s):
            if isinstance(val, str) and val.strip():
                texts.append((f'summaries:{i}', val.strip()))

    # en_translated_summaries is usually a dict: lang -> {'text': ...} or lang -> str
    ts = obj.get('en_translated_summaries')
    if isinstance(ts, dict):
        for lang, val in ts.items():
            if isinstance(val, dict):
                txt = val.get('text')
                if isinstance(txt, str) and txt.strip():
                    texts.append((f'en_translated_summaries:{lang}', txt.strip()))
            elif isinstance(val, str) and val.strip():
                texts.append((f'en_translated_summaries:{lang}', val.strip()))
    elif isinstance(ts, list):
        for i, val in enumerate(ts):
            if isinstance(val, str) and val.strip():
                texts.append((f'en_translated_summaries:{i}', val.strip()))
            elif isinstance(val, dict):
                txt = val.get('text')
                if isinstance(txt, str) and txt.strip():
                    texts.append((f'en_translated_summaries:{i}', txt.strip()))

    # de-duplicate by text while preserving order
    out = []
    seen = set()
    for source, txt in texts:
        key = txt.strip()
        if key in seen:
            continue
        seen.add(key)
        out.append((source, key))
    return out

records = []
for wid in dev_ids['wikidata_id'].tolist():
    obj = load_tma_summary_json(wid)
    if not isinstance(obj, dict):
        continue

    variants = collect_tma_texts(obj)
    if len(variants) < 2:
        continue

    for j, (src_name, txt) in enumerate(variants):
        records.append({
            'dataset': 'tell_me_again',
            'group_id': wid,
            'story_id': f'{wid}_{j}',
            'story_text': txt,
            'title': obj.get('title_en') or obj.get('title') or '',
            'source_variant': src_name,
        })

tma_stories_df = pd.DataFrame(records)
expected_cols = ['dataset', 'group_id', 'story_id', 'story_text', 'title', 'source_variant']
if tma_stories_df.empty:
    tma_stories_df = pd.DataFrame(columns=expected_cols)

print('dev_ids:', len(dev_ids))
print('usable story snippets:', len(tma_stories_df))
if tma_stories_df.empty:
    print('groups with >=2 retellings: 0 (no usable dev stories found)')
else:
    print('groups with >=2 retellings:', tma_stories_df.groupby('group_id').size().ge(2).sum())
    print('avg variants per group:', round(float(tma_stories_df.groupby('group_id').size().mean()), 2))
tma_stories_df.head(3)

dev_ids: 2950
usable story snippets: 19247
groups with >=2 retellings: 2949
avg variants per group: 6.53


,dataset,group_id,story_id,story_text,title,source_variant
0,tell_me_again,753610,753610_0,Ruby is a woman in her early 20s and the narra...,Ruby in Paradise,summaries:en
1,tell_me_again,753610,753610_1,Ruby quitte le Tennessee pour Panama City (Flo...,Ruby in Paradise,summaries:fr
2,tell_me_again,753610,753610_2,Ruby (Judd) es una joven que deja su pequeño p...,Ruby in Paradise,summaries:es


## Explore Movie Remakes (dev split if present, else fallback)


In [5]:
mr_dir = CONFIG['movie_remakes_dir']
instances_csv = mr_dir / 'testInstances.csv'
clean_tsv = mr_dir / 'movieRemakesManuallyCleaned.tsv'

if not mr_dir.exists():
    raise FileNotFoundError(f'movie remakes folder not found: {mr_dir}')
if not instances_csv.exists():
    raise FileNotFoundError(f'testInstances.csv not found: {instances_csv}')
if not clean_tsv.exists():
    raise FileNotFoundError(f'movieRemakesManuallyCleaned.tsv not found: {clean_tsv}')

inst = pd.read_csv(instances_csv)
inst.columns = [c.strip() for c in inst.columns]
# Expected columns after strip: clusterid, movieid
if 'clusterid' not in inst.columns or 'movieid' not in inst.columns:
    raise ValueError(f'Unexpected columns in {instances_csv}: {list(inst.columns)}')

inst['clusterid'] = inst['clusterid'].astype(str).str.strip()
inst['movieid'] = inst['movieid'].astype(str).str.strip()
valid_movie_ids = set(inst['movieid'])

rows = []
with clean_tsv.open('r', encoding='utf-8', errors='ignore') as f:
    for line in f:
        line = line.rstrip('\n')
        if not line:
            continue
        cols = line.split('\t')
        # Format: cluster_id, then repeated triplets (movieid, title, summary)
        if len(cols) < 4:
            continue
        cluster_id = cols[0].strip()
        for i in range(1, len(cols) - 2, 3):
            movie_id = cols[i].strip()
            title = cols[i + 1].strip()
            summary = cols[i + 2].strip()
            if not movie_id or not summary:
                continue
            # Keep only movie IDs in testInstances to match benchmark split source
            if movie_id not in valid_movie_ids:
                continue
            rows.append({
                'dataset': 'movie_remakes',
                'group_id': cluster_id,
                'story_id': movie_id,
                'story_text': summary,
                'title': title,
            })

mr_stories_df = pd.DataFrame(rows)
expected_cols = ['dataset', 'group_id', 'story_id', 'story_text', 'title']
if mr_stories_df.empty:
    mr_stories_df = pd.DataFrame(columns=expected_cols)
else:
    mr_stories_df = (
        mr_stories_df.drop_duplicates(subset=['story_id'])
        .assign(group_id=lambda d: d['group_id'].astype(str))
        .copy()
    )

print('instances rows:', len(inst))
print('usable clean summaries:', len(mr_stories_df))
if not mr_stories_df.empty and 'group_id' in mr_stories_df.columns:
    print('clusters with >=2 movies:', mr_stories_df.groupby('group_id').size().ge(2).sum())
else:
    print('clusters with >=2 movies: 0 (no usable stories found)')
mr_stories_df.head(3)


instances rows: 466
usable clean summaries: 466
clusters with >=2 movies: 181


,dataset,group_id,story_id,story_text,title
0,movie_remakes,1,14141235,The jury decides whether a young Chechen boy i...,12_(2007_film)
1,movie_remakes,1,11081144,After the final closing arguments have been pr...,12_Angry_Men_(1997_film)
2,movie_remakes,1,11094452,The story begins in a courtroom where a teenag...,Ek_Ruka_Hua_Faisla


## Pair Builders


In [6]:
def sample_positive_pairs(stories_df: pd.DataFrame, n_pairs: int, rng: random.Random):
    """
    Positive pairs: same group_id, different story_id.
    """
    candidates = []
    for gid, grp in stories_df.groupby('group_id'):
        ids = grp['story_id'].tolist()
        for i in range(len(ids)):
            for j in range(i + 1, len(ids)):
                a = grp.iloc[i]
                b = grp.iloc[j]
                candidates.append({
                    'dataset': a['dataset'],
                    'pair_type': 'positive',
                    'group_a': a['group_id'],
                    'group_b': b['group_id'],
                    'story_id_a': a['story_id'],
                    'story_id_b': b['story_id'],
                    'story_text_a': a['story_text'],
                    'story_text_b': b['story_text'],
                    'label': 1,
                })
    rng.shuffle(candidates)
    return candidates[:min(n_pairs, len(candidates))], len(candidates)


def sample_negative_pairs(stories_df: pd.DataFrame, n_pairs: int, rng: random.Random, candidates_df: pd.DataFrame | None = None):
    """
    Random negatives: different group_id.
    """
    base_df = candidates_df if candidates_df is not None else stories_df
    rows = base_df.to_dict(orient='records')
    if len(rows) < 2:
        return [], 0

    pair_set = set()
    out = []
    max_attempts = n_pairs * 60
    attempts = 0

    while len(out) < n_pairs and attempts < max_attempts:
        attempts += 1
        a, b = rng.sample(rows, 2)
        if a['group_id'] == b['group_id']:
            continue

        key = tuple(sorted([a['story_id'], b['story_id']]))
        if key in pair_set:
            continue
        pair_set.add(key)

        out.append({
            'dataset': a['dataset'],
            'pair_type': 'random_negative',
            'group_a': a['group_id'],
            'group_b': b['group_id'],
            'story_id_a': a['story_id'],
            'story_id_b': b['story_id'],
            'story_text_a': a['story_text'],
            'story_text_b': b['story_text'],
            'label': 0,
        })

    return out, len(out)


def build_eval_pairs(stories_df: pd.DataFrame, total_pairs: int, positive_ratio: float, seed: int, negative_candidates_df: pd.DataFrame | None = None):
    rng = random.Random(seed)
    n_pos = int(total_pairs * positive_ratio)
    n_neg = total_pairs - n_pos

    pos_pairs, pos_pool = sample_positive_pairs(stories_df, n_pos, rng)
    neg_pairs, neg_actual = sample_negative_pairs(stories_df, n_neg, rng, candidates_df=negative_candidates_df)

    built = pos_pairs + neg_pairs
    rng.shuffle(built)

    meta = {
        'requested_total': total_pairs,
        'requested_pos': n_pos,
        'requested_neg': n_neg,
        'built_total': len(built),
        'built_pos': sum(1 for x in built if x['label'] == 1),
        'built_neg': sum(1 for x in built if x['label'] == 0),
        'positive_pool_size': pos_pool,
        'negatives_built': neg_actual,
    }
    return built, meta

## Build 100 Tell-Me-Again + 100 Movie-Remakes Pairs


In [7]:
# Tell-Me-Again: keep positive sampling unchanged, but restrict negatives to English-text variants only.
# English-text variants are either raw English summaries or translated-to-English summaries.
tma_english_mask = tma_stories_df['source_variant'].astype(str).str.startswith('summaries:en') |                    tma_stories_df['source_variant'].astype(str).str.startswith('en_translated_summaries:')
tma_negative_candidates_df = tma_stories_df[tma_english_mask].copy()

print('Tell Me Again total variants:', len(tma_stories_df))
print('Tell Me Again English candidates for negatives:', len(tma_negative_candidates_df))

tma_pairs, tma_meta = build_eval_pairs(
    tma_stories_df,
    total_pairs=CONFIG['tell_me_again_total_pairs'],
    positive_ratio=CONFIG['positive_ratio'],
    seed=CONFIG['seed'],
    negative_candidates_df=tma_negative_candidates_df,
)

mr_pairs, mr_meta = build_eval_pairs(
    mr_stories_df,
    total_pairs=CONFIG['movie_remakes_total_pairs'],
    positive_ratio=CONFIG['positive_ratio'],
    seed=CONFIG['seed'] + 7,
)

print('Tell Me Again meta:', tma_meta)
print('Movie Remakes meta:', mr_meta)


Tell Me Again total variants: 19247
Tell Me Again English candidates for negatives: 10936
Tell Me Again meta: {'requested_total': 100, 'requested_pos': 50, 'requested_neg': 50, 'built_total': 100, 'built_pos': 50, 'built_neg': 50, 'positive_pool_size': 60376, 'negatives_built': 50}
Movie Remakes meta: {'requested_total': 100, 'requested_pos': 50, 'requested_neg': 50, 'built_total': 100, 'built_pos': 50, 'built_neg': 50, 'positive_pool_size': 251, 'negatives_built': 50}


## Combine, Inspect, and Save


In [8]:
all_pairs = tma_pairs + mr_pairs
for i, p in enumerate(all_pairs):
    p['pair_id'] = f"{p['dataset']}__{i:04d}"

pairs_df = pd.DataFrame(all_pairs)

print('Total pairs:', len(pairs_df))
print('By dataset:')
print(pairs_df['dataset'].value_counts().to_string())
print('By dataset x pair_type:')
print(pairs_df.groupby(['dataset', 'pair_type']).size().to_string())
pairs_df[['pair_id', 'dataset', 'pair_type', 'story_id_a', 'story_id_b', 'label']].head(10)

Total pairs: 200
By dataset:
dataset
tell_me_again    100
movie_remakes    100
By dataset x pair_type:
dataset        pair_type      
movie_remakes  positive           50
               random_negative    50
tell_me_again  positive           50
               random_negative    50


,pair_id,dataset,pair_type,story_id_a,story_id_b,label
0,tell_me_again__0000,tell_me_again,random_negative,951587_5,19839914_0,0
1,tell_me_again__0001,tell_me_again,positive,741737_2,741737_4,1
2,tell_me_again__0002,tell_me_again,positive,759467_4,759467_5,1
3,tell_me_again__0003,tell_me_again,random_negative,927048_0,1212396_6,0
4,tell_me_again__0004,tell_me_again,positive,1215110_4,1215110_6,1
5,tell_me_again__0005,tell_me_again,positive,21528262_4,21528262_6,1
6,tell_me_again__0006,tell_me_again,positive,1210832_2,1210832_6,1
7,tell_me_again__0007,tell_me_again,positive,2893887_1,2893887_7,1
8,tell_me_again__0008,tell_me_again,random_negative,1076912_6,18225084_6,0
9,tell_me_again__0009,tell_me_again,random_negative,15676054_8,1452673_2,0


In [9]:
CONFIG['output_json'].parent.mkdir(parents=True, exist_ok=True)

# JSON is best for long texts; CSV is also saved for convenience.
pairs_df.to_json(CONFIG['output_json'], orient='records', indent=2, force_ascii=False)
pairs_df.to_csv(CONFIG['output_csv'], index=False)

print('Saved JSON:', CONFIG['output_json'])
print('Saved CSV :', CONFIG['output_csv'])

Saved JSON: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.json
Saved CSV : /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv


## Notes

- `label=1` means true retelling/remake pair (positive).
- `label=0` means random cross-group pair (negative).
- To change counts later, edit `tell_me_again_total_pairs` and `movie_remakes_total_pairs` in `CONFIG`.
- For cosine eval later, you can embed `story_text_a` and `story_text_b` and compute similarity per `pair_id`.


In [10]:
# Quick sanity check: load eval_data CSV and print head/basic stats.
from pathlib import Path

csv_path = DATA_DIR / 'eval_data' / 'eval_story_pairs_200.csv'
if not csv_path.exists():
    raise FileNotFoundError(f'CSV not found at: {csv_path}')

eval_df = pd.read_csv(csv_path)

print('Loaded:', csv_path)
print('Shape:', eval_df.shape)
print('\nHead:')
print(eval_df.head().to_string(index=False))

print('\nBasic stats:')
if 'dataset' in eval_df.columns:
    print('By dataset:')
    print(eval_df['dataset'].value_counts(dropna=False).to_string())

if 'pair_type' in eval_df.columns:
    print('\nBy pair_type:')
    print(eval_df['pair_type'].value_counts(dropna=False).to_string())

if {'dataset', 'pair_type'}.issubset(eval_df.columns):
    print('\nBy dataset x pair_type:')
    print(eval_df.groupby(['dataset', 'pair_type']).size().to_string())

if 'label' in eval_df.columns:
    print('\nLabel distribution:')
    print(eval_df['label'].value_counts(dropna=False).to_string())

for col in ['story_text_a', 'story_text_b']:
    if col in eval_df.columns:
        lengths = eval_df[col].fillna('').astype(str).str.len()
        print(f"\n{col} length stats:")
        print(lengths.describe().to_string())



Loaded: /Users/shayan/Projects/NarrativeSimilarity/data/eval_data/eval_story_pairs_200.csv
Shape: (200, 10)

Head:
      dataset       pair_type  group_a  group_b story_id_a story_id_b                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

In [13]:
eval_df.head(20)

,dataset,pair_type,group_a,group_b,story_id_a,story_id_b,story_text_a,story_text_b,label,pair_id
0,tell_me_again,random_negative,951587,19839914,951587_5,19839914_0,"Kathy, a 31-year-old woman, reflects on her pa...","Billy Lynn, a 19-year-old US Army specialist f...",0,tell_me_again__0000
1,tell_me_again,positive,741737,741737,741737_2,741737_4,"Milo Boyd (Gerard Butler), un cazarrecompensas...",Milo Boyd è un ex detective del dipartimento d...,1,tell_me_again__0001
2,tell_me_again,positive,759467,759467,759467_4,759467_5,"Colin Lamb, biologo marino e anche agente del ...",On his way to visit the settlement on Wilbraha...,1,tell_me_again__0002
3,tell_me_again,random_negative,927048,1212396,927048_0,1212396_6,"While scrubbing the floor at home, Belle is i...","Bertha Thompson's father, pressed by her emplo...",0,tell_me_again__0003
4,tell_me_again,positive,1215110,1215110,1215110_4,1215110_6,"Raven, killer a pagamento, viene assoldato da ...","A contract killer, Phillip Raven (Alan Ladd) s...",1,tell_me_again__0004
5,tell_me_again,positive,21528262,21528262,21528262_4,21528262_6,"Dr. Jan Żabiński and his wife, Antonina, run t...",It is the true story about the caretakers of t...,1,tell_me_again__0005
6,tell_me_again,positive,1210832,1210832,1210832_2,1210832_6,La historia comienza en una ciudad centroeurop...,"In a city besieged by the Turks, a play tells ...",1,tell_me_again__0006
7,tell_me_again,positive,2893887,2893887,2893887_1,2893887_7,"Durant l'été 1977, le jeune Scott Thorson (en)...",It's 1977. After a difficult childhood and ado...,1,tell_me_again__0007
8,tell_me_again,random_negative,1076912,18225084,1076912_6,18225084_6,The action takes place under the Rashō gate fr...,# General presentation\nA few years after the ...,0,tell_me_again__0008
9,tell_me_again,random_negative,15676054,1452673,15676054_8,1452673_2,After making a mistake that cost the life of o...,Black Bill is a mysterious vigilante who rides...,0,tell_me_again__0009
